# Feasibility Pump

Algoritmo introducido por el año 2005. La idea es construir dos secuencias de puntos que eventualmente convergan a una solución factible para un problema de optimización. 

Una secuencia consiste en los puntos factibles para una relajación continua mientras que la otra corresponde a una solución entera que llegar a violar algunas restricciones.

Primero, resolvemos el LP del problema relajado para el nodo actual:

$$\bar{x}=\arg\min\lbrace c^Tx\;:\;x\in P\rbrace$$

A partir de tal punto, empieza el algoritmo iterativo: 

1. Si $\bar{x}$ es entero, entonces hemos terminado.
2. De lo contrario, nos definimos $\tilde{x}$ como un redondeo al entero más cercano de $\bar{x}$.
3. Si detectamos un ciclo, entonces perturbamos $\tilde{x}$.
4. Luego, nos definimos $\bar{x}$ como la proyección lineal de $\tilde{x}$. En estricto rigor, la proyección lineal llega a ser la solución del problema

$$x = \arg\min\lvert\lvert \bar{x}-\tilde{x}\rvert\rvert\;\;Ax\leq b$$

La solución entera del algoritmo iterativo llegaría a ser una solución factible al MIP. 

Veamos como se ve esto con una instancia knapsack. Notemos que tendremos que resolver problemas de optimización dentro de los branches, por lo tanto, se necesitará hacer el "truco" del auxiliar pasado.

In [43]:
import collections
import math
import random
import time

import gurobipy as gp
from gurobipy import GRB

from helpers import build_var_index, cache_linear_rows_for

Primero, nos definimos una función para poder clonar el modelo actual como un LP

In [44]:
def clone_as_lp(model, xvars):
    """
    Clona el modelo como LP: todas las variables pasan a continuas.
    Devuelve (lp, var_map) donde var_map: original_var -> cloned_var
    """
    lp = model.copy()
    lp.Params.OutputFlag = 0
    lp.Params.Presolve = 2
    # vuelve todo continuo
    for v in lp.getVars():
        v.VType = GRB.CONTINUOUS
    lp.update()

    # Pequeño arreglo por si los índices no coinciden
    name_to_lpvar = {v.VarName: v for v in lp.getVars()}
    var_map = {v: name_to_lpvar[v.VarName] for v in xvars}
    return lp, var_map

Ahora, notemos que, para resolver un LP que minimiza una norma, hay que crearnos el siguiente cambio de variable:

$$
\min \lvert\lvert x - \bar{x}\rvert\rvert \iff \min \Phi\;\;\text{s.a}\;\Phi = \lvert\lvert x-\bar{x}\rvert\rvert$$

Por lo mismo, nos definiremos una función que haga tal trabajo para una norma $L_1$ y con desigualdades. 


In [45]:
def build_distance_layer(lp, x_lp_vars, name_prefix="fp"):
    """
    Crea variables d_j >= |x_j - z_j| linealizado:
        d_j - x_j >= -z_j
        d_j + x_j >=  z_j
    (z_j es constante que se actualiza en RHS cada iteración)
    Devuelve (d_vars, cons1, cons2), donde cons1,cons2 son listas de constrs por j.
    """
    d_vars = []
    cons1 = []  # d_j - x_j >= -z_j   (RHS = -z_j)
    cons2 = []  # d_j + x_j >=  z_j   (RHS =  z_j)
    for j, xj in enumerate(x_lp_vars):
        dj = lp.addVar(lb=0.0, name=f"{name_prefix}_d[{j}]")
        d_vars.append(dj)
        c1 = lp.addConstr(
            dj - xj >= 0.0, name=f"{name_prefix}_c1[{j}]"
        )  # RHS se ajustará
        c2 = lp.addConstr(
            dj + xj >= 0.0, name=f"{name_prefix}_c2[{j}]"
        )  # RHS se ajustará
        cons1.append(c1)
        cons2.append(c2)
    lp.update()
    # Objetivo: min sum d_j  (lo actualizaremos si quieres ponderar con el original)
    lp.setObjective(gp.quicksum(d_vars), GRB.MINIMIZE)
    lp.update()
    return d_vars, cons1, cons2

Ahora, nos definimos una función helper para actualizar el *"right hand side"*.

In [46]:
def update_z_rhs(lp, cons1, cons2, z):
    """
    Actualiza RHS de:
      d - x >= -z  (RHS = -z)
      d + x >=  z  (RHS =  z)
    """
    for j in range(len(z)):
        cons1[j].RHS = -float(z[j])
        cons2[j].RHS = float(z[j])
    lp.update()

Otra función para redondear al entero más cercano. Esta función es bien importante, ya que si queremos imponer mejoras al proceso de redondeo, necesitaremos modificar esta función

In [47]:
def round_ints(x_vals, vtypes, eps=1e-6, seed=None):
    """
    Redondeo "cercano" para enteras y binaria; tie-break aleatorio si muy cerca de .5
    Devuelve z (solo para variables enteras/binary; para continuas usamos x_vals
    """
    if seed is not None:
        random.seed(seed)
    z = [None] * len(x_vals)
    for j, (xj, t) in enumerate(zip(x_vals, vtypes)):
        if t in (GRB.BINARY, GRB.INTEGER):
            r = round(xj)
            # tie-break suave alrededor de .5
            if abs(xj - 0.5) <= 1e-9 and t == GRB.BINARY:
                r = 1 if random.random() < 0.5 else 0
            z[j] = float(r)
        else:
            z[j] = float(xj)
    return z

Nos definimos otra función para que nos pueda decir la norma $L_1$.

In [48]:
def distance_L1(x_vals, z, mask_int):
    """||x - z||_1 sobre componentes enteras/binarias (mask_int=True)."""
    return sum(abs(x_vals[j] - z[j]) for j in range(len(x_vals)) if mask_int[j])

Otra para poder hacer una leve perturbación que pueda romper ciclos infinitos.

In [49]:
def perturb_z(z, vtypes, num_flips=1, frac_order=None, seed=None):
    """
    Anticiclo: voltea unas pocas variables (enteras/binary).
    Si se provee frac_order (índices ordenados por fraccionalidad previa), usa esos.
    """
    if seed is not None:
        random.seed(seed)
    idxs = [j for j, t in enumerate(vtypes) if t in (GRB.BINARY, GRB.INTEGER)]
    if not idxs:
        return z[:]
    candidates = frac_order if frac_order is not None else idxs
    z_new = z[:]
    flips = 0
    for j in candidates:
        if vtypes[j] == GRB.BINARY:
            z_new[j] = 1.0 - z_new[j]
        else:
            # para enteras, +1 o -1 aleatorio
            z_new[j] = z_new[j] + (1 if random.random() < 0.5 else -1)
        flips += 1
        if flips >= num_flips:
            break
    return z_new

Una función para verificar factibilidad del problema del MIP

In [50]:
def check_feasible_by_fixing_integers(orig_model, xvars, z_int, time_limit=5.0):
    """
    Verifica factibilidad fuerte: clona modelo original, fija enteras a z_int,
    optimiza como LP factible (objetivo 0).
    Devuelve (is_feasible, sol_values_dict) si factible.
    """
    mm = orig_model.copy()
    mm.Params.OutputFlag = 0
    mm.Params.TimeLimit = time_limit
    # Fijar enteras:
    name_to_var = {v.VarName: v for v in mm.getVars()}
    for v, val in zip(xvars, z_int):
        w = name_to_var[v.VarName]
        if w.VType in (GRB.BINARY, GRB.INTEGER):
            w.lb = w.ub = float(val)
        else:
            # continuas no se fijan
            pass
    mm.update()
    # objetivo 0
    mm.setObjective(0.0, GRB.MINIMIZE)
    mm.optimize()
    if mm.status == GRB.OPTIMAL or mm.status == GRB.SUBOPTIMAL:
        # extraer solución
        sol = {var.VarName: var.X for var in mm.getVars()}
        return True, sol
    return False, {}

Y, ahora si, nos definimos nuestra función "main" para el feasibility pump que recibe una semilla $x_0$.

In [51]:
def feasibility_pump_seeded(
    model,
    xvars,
    x0,  # semilla inicial (solución LP relajada del callback)
    max_iters=40,  # número máximo de iteraciones FP
    time_limit=10.0,  # tiempo máximo total (segundos)
    int_eps=1e-6,  # tolerancia para redondeo de enteras
    seed=0,  # semilla aleatoria
    verbose=False,
):  # imprimir info detallada o no
    # Marca de tiempo inicial (para cortar por tiempo)
    t0 = time.time()
    # Inicializa semilla aleatoria (para redondeo y perturbaciones)
    random.seed(seed)

    # Tipos de variables (BINARY, INTEGER o CONTINUOUS)
    vtypes = [v.VType for v in xvars]
    # Máscara booleana: True si variable es entera o binaria
    mask_int = [t in (GRB.BINARY, GRB.INTEGER) for t in vtypes]

    # === 1) Construcción del LP auxiliar para la proyección ===
    # Clona el modelo original pero cambia todas las variables a continuas
    lp, var_map = clone_as_lp(model, xvars)
    # Obtiene las variables clonadas en el mismo orden
    x_lp_vars = [var_map[v] for v in xvars]
    # Añade una capa de “distancia” (d_j >= |x_j - z_j|) que se actualizará en cada iteración
    d_vars, cons1, cons2 = build_distance_layer(lp, x_lp_vars, name_prefix="fp")

    # === 2) Inicialización ===
    # Redondeo inicial: convierte la solución LP (x0) en entera z
    z = round_ints(x0, vtypes, eps=int_eps, seed=seed)
    # Calcula la distancia L1 inicial entre x0 y z (solo en componentes enteras)
    best_L1 = distance_L1(x0, z, mask_int)
    if verbose:
        print(f"[FP-seeded] it=0  ||x-z||_1(int)={best_L1:.6g}")

    # Actualiza los RHS (lados derechos) de las restricciones d - x >= -z y d + x >= z
    update_z_rhs(lp, cons1, cons2, z)

    # Guarda los redondeos anteriores para evitar ciclos
    z_history = set()
    # Convierte z a tupla hashable (solo enteras) para registrar la configuración actual
    z_key = tuple(int(zj) if mask_int[j] else 0 for j, zj in enumerate(z))
    z_history.add(z_key)

    # Contadores de iteración y sin mejora
    it, no_improve = 0, 0

    # === 3) Bucle principal del Feasibility Pump ===
    while it < max_iters and (time.time() - t0) < time_limit:
        it += 1

        # (a) Minimiza la distancia ||x - z||_1 en el LP
        lp.optimize()
        # Si el LP no es óptimo o factible → se detiene
        if lp.status not in (GRB.OPTIMAL, GRB.SUBOPTIMAL):
            if verbose:
                print(f"[FP-seeded] LP infactible en it={it}.")
            break

        # (b) Extrae la solución relajada x (continuas)
        x_rel = [xj.X for xj in x_lp_vars]
        # (c) Redondea nuevamente x_rel → z_new
        z_new = round_ints(x_rel, vtypes, eps=int_eps, seed=seed + it)
        # (d) Calcula nueva distancia ||x - z||_1 sobre enteras
        L1 = distance_L1(x_rel, z_new, mask_int)
        if verbose:
            print(f"[FP-seeded] it={it}  ||x-z||_1(int)={L1:.6g}")

        # (e) Actualiza mejor distancia si mejora
        if L1 + 1e-12 < best_L1:
            best_L1 = L1
            no_improve = 0
        else:
            no_improve += 1

        # === 4) Chequeo de factibilidad entera ===
        # Si todas las enteras de x_rel son (casi) enteras, prueba factibilidad total
        if all(
            (not mask_int[j]) or abs(x_rel[j] - round(x_rel[j])) <= 1e-8
            for j in range(len(x_rel))
        ):
            ok, sol = check_feasible_by_fixing_integers(
                model,
                xvars,
                [
                    round(x_rel[j]) if mask_int[j] else x_rel[j]
                    for j in range(len(x_rel))
                ],
            )
            # Si se logra factibilidad → devuelve solución
            if ok:
                # Calcula valor objetivo si el modelo tiene vector de costos _v
                obj_val = (
                    sum(model._v[i] * sol[f"x[{i}]"] for i in range(len(model._v)))
                    if hasattr(model, "_v")
                    else None
                )

                if verbose:
                    print(f"[FP-seeded] solución factible en it={it}.")
                    if obj_val is not None:
                        print(
                            f"[FP-seeded] valor objetivo de la solución propuesta: {obj_val:.3f}"
                        )
                    # Si hay incumbente en el modelo, compara objetivos
                    if model.SolCount > 0:
                        best = model.ObjVal
                        print(f"[FP-seeded] incumbente actual: {best:.3f}")
                        print(
                            f"[FP-seeded] {'MEJORA!' if obj_val > best else 'no mejora.'}"
                        )
                # Devuelve el diccionario con valores de la solución
                return sol

        # === 5) Anticiclo o perturbación ===
        # Si repetimos el mismo z o no mejoramos durante varias iteraciones,
        # se introduce una perturbación aleatoria
        z_key = tuple(int(z_new[j]) if mask_int[j] else 0 for j in range(len(z_new)))
        if (z_key in z_history) or (no_improve >= 3):
            # Ordena las variables por fraccionalidad (mayor primero)
            fracs = sorted(
                [
                    (j, abs(x_rel[j] - round(x_rel[j])))
                    for j in range(len(x_rel))
                    if mask_int[j]
                ],
                key=lambda t: t[1],
                reverse=True,
            )
            frac_order = [j for j, _ in fracs]
            # Decide cuántas variables “voltear” para romper el ciclo
            flips = 2 if len(frac_order) >= 2 else 1
            if verbose:
                print(f"[FP-seeded] perturbación (flips={flips}) en it={it}.")
            # Aplica la perturbación en z_new
            z_new = perturb_z(
                z_new,
                vtypes,
                num_flips=flips,
                frac_order=frac_order,
                seed=seed + 1234 + it,
            )
            # Reinicia el contador de no mejora
            no_improve = 0

        # (f) Registra la configuración actual
        z_history.add(z_key)
        # (g) Actualiza z y RHS de las restricciones para la siguiente iteración
        z = z_new
        update_z_rhs(lp, cons1, cons2, z)

    # === 6) Si se agota el tiempo o las iteraciones ===
    if verbose:
        print("[FP-seeded] sin solución factible en límites dados.")
    return None

Ahora si, podemos definirnos la función del callback y nuestra instancia.

In [52]:
def make_feaspump_rounds_callback(model, xvars, collect_every_k_nodes=1, pool_cap=5000):
    """
    - Recolecta soluciones LP del nodo en model._fp_pool (saco).
    - Inyecta soluciones enteras en cola model._fp_inject (deque) con cbSetSolution().
    - Guarda historial de LB/UB/nodes si están los arrays en el modelo.
    """
    model.update()
    if not hasattr(model, "_fp_pool"):
        model._fp_pool = []
    if not hasattr(model, "_fp_inject"):
        model._fp_inject = collections.deque()
    if not hasattr(model, "_nodes_hist"):
        model._nodes_hist, model._lb_hist, model._ub_hist = [], [], []

    model._cb_calls = 0

    def cb(cb_model, where):
        # Métricas globales
        if where == GRB.Callback.MIP:
            try:
                nodes = cb_model.cbGet(GRB.Callback.MIP_NODCNT)
                bestbd = cb_model.cbGet(GRB.Callback.MIP_OBJBND)
                bestst = cb_model.cbGet(GRB.Callback.MIP_OBJBST)
                sense = model.ModelSense
                LB, UB = (bestst, bestbd) if sense == -1 else (bestbd, bestst)
                model._nodes_hist.append(nodes)
                model._lb_hist.append(LB)
                model._ub_hist.append(UB)
            except gp.GurobiError:
                pass
            return

        if where != GRB.Callback.MIPNODE:
            return

        status = cb_model.cbGet(GRB.Callback.MIPNODE_STATUS)
        if status != GRB.OPTIMAL:
            return

        # Inyectar solución entera si hay en cola
        if model._fp_inject:
            try:
                sol = model._fp_inject[0]  # peek
                cb_model.cbSetSolution(xvars, sol)
                model._fp_inject.popleft()  # consumimos una
            except gp.GurobiError:
                pass  # si Gurobi no la acepta, seguimos

        # Recolectar relajaciones
        model._cb_calls += 1
        if collect_every_k_nodes > 1 and (model._cb_calls % collect_every_k_nodes != 0):
            return
        if len(model._fp_pool) >= pool_cap:
            return

        try:
            x_rel = cb_model.cbGetNodeRel(xvars)
            model._fp_pool.append(list(x_rel))
        except gp.GurobiError:
            return

    return cb

Y, nuestra función "main" (el driver)

In [56]:
def run_bnb_with_fp_rounds(
    model,
    xvars,
    rounds=5,  # Nº de rondas de B&B
    nodes_per_round=5000,  # Límite de nodos por ronda
    collect_every_k_nodes=1,  # Frecuencia de muestreo de relajaciones en el callback
    pool_cap=5000,  # Máximo de relajaciones a guardar
    seeds_per_round=3,  # Nº de semillas FP a probar por ronda
    fp_max_iters=40,  # Iteraciones máximas del FP
    fp_time_limit=10.0,  # Tiempo máximo del FP
    inject_as_start=True,  # Si True, setea v.start con la solución FP
    fp_verbose=False,  # Verbosidad interna del FP
    driver_verbose=True,  # Verbosidad de este driver
    presolve_fp=True,  # Ejecutar FP antes de B&B usando la raíz
    presolve_seeds=3,  # Nº de semillas FP en presolve
    presolve_try_round_pert=True,  # Generar semillas perturbadas en presolve
    presolve_time_limit=None,  # Límite de tiempo del FP en presolve (si None usa fp_time_limit)
):
    """
    Orquesta B&B intercalando Feasibility Pump (FP):
    - (Opcional) Presolve: FP seed-eado desde la relajación LP de la raíz.
    - B&B por rondas con NodeLimit; entre rondas ejecuta FP con semillas del pool
      recolectado por el callback.
    - Inyecta soluciones enteras (cbSetSolution) y/o como MIP start (v.start).
    """

    # Construye el callback que:
    #  - guarda relajaciones (en model._fp_pool)
    #  - consume/inyecta soluciones enteras (en model._fp_inject)
    cb = make_feaspump_rounds_callback(
        model, xvars, collect_every_k_nodes=collect_every_k_nodes, pool_cap=pool_cap
    )

    # Asegura que existan las estructuras que usa el callback
    if not hasattr(model, "_fp_inject"):
        model._fp_inject = []  # cola de soluciones enteras listas para inyectar
    if not hasattr(model, "_fp_pool"):
        model._fp_pool = []  # pool (saco) de relajaciones LP recolectadas

    # Guarda valores originales para restaurar al final (buen ciudadano)
    orig_NodeLimit = model.Params.NodeLimit
    orig_TimeLimit = model.Params.TimeLimit

    # Precomputa tipos y máscara de variables enteras/binarias
    vtypes = [v.VType for v in xvars]
    mask_int = [t in (GRB.BINARY, GRB.INTEGER) for t in vtypes]

    # Métrica auxiliar: distancia a integridad (sólo sobre componentes enteras/binarias)
    def frac_distance(x):
        return sum(abs(x[j] - round(x[j])) for j in range(len(x)) if mask_int[j])

    # ================= helpers internos =================

    def _root_relax_and_seeds():
        """Resuelve la relajación LP de la raíz y arma semillas para FP."""
        # Clona el modelo como LP (todas las variables continuas)
        lp_relax, var_map = clone_as_lp(model, xvars)
        # Variables x del LP clonado en el mismo orden que xvars
        x_lp_vars = [var_map[v] for v in xvars]
        # Resuelve la relajación
        lp_relax.optimize()
        # Si LP no es solucionable, no hay semillas
        if lp_relax.status not in (GRB.OPTIMAL, GRB.SUBOPTIMAL):
            return []

        # x0: solución de la relajación raíz
        x0 = [v.X for v in x_lp_vars]

        # Inicializa lista de semillas con la solución raíz
        seeds = [x0]

        # Si se desea, genera semillas adicionales con pequeñas perturbaciones
        if presolve_try_round_pert and presolve_seeds > 1:
            # Ordena índices por fraccionalidad (mayor primero)
            fracs = [
                (j, abs(x0[j] - round(x0[j]))) for j in range(len(x0)) if mask_int[j]
            ]
            fracs.sort(key=lambda t: t[1], reverse=True)
            # Toma un top de más fraccionales para focalizar las perturbaciones
            top_idx = [j for j, _ in fracs[: max(3, min(10, len(fracs)))]]
            rng = random.Random(12345)
            # Genera hasta presolve_seeds-1 semillas nuevas
            for s in range(1, presolve_seeds):
                xd = x0[:]  # copia de la seed base
                for j in top_idx:
                    if vtypes[j] == GRB.BINARY:
                        # Flip ocasional para explorar vecindarios
                        if rng.random() < 0.35:
                            xd[j] = 1.0 - round(x0[j])
                        else:
                            xd[j] = round(x0[j])
                    else:
                        # En enteras generales, empuja a entero cercano (aquí simple)
                        r = round(x0[j])
                        if rng.random() < 0.5:
                            xd[j] = r
                        else:
                            xd[j] = r
                seeds.append(xd)
        return seeds

    # =============== bloque principal con limpieza garantizada ===============
    try:
        # -------------------- PRESOLVE FP (opcional) --------------------
        if presolve_fp:
            if driver_verbose:
                print("\n=== Presolve FP: usando relajación LP raíz ===")

            # Asegura al menos 1 semilla
            presolve_seeds = max(1, presolve_seeds)

            # Obtiene semillas desde la relajación LP
            root_seeds = _root_relax_and_seeds()

            best_sol_vec = None  # almacenará la primera solución FP factible

            # Itera sobre las primeras 'presolve_seeds' semillas disponibles
            for k, x0 in enumerate(root_seeds[:presolve_seeds], 1):
                if driver_verbose:
                    print(
                        f"[Presolve] FP seed {k}/{min(presolve_seeds,len(root_seeds))}  "
                        f"||frac||={frac_distance(x0):.3f}"
                    )

                # Ejecuta FP con esa semilla
                sol_dict = feasibility_pump_seeded(
                    model,
                    xvars,
                    x0,
                    max_iters=fp_max_iters,
                    time_limit=(
                        presolve_time_limit
                        if presolve_time_limit is not None
                        else fp_time_limit
                    ),
                    int_eps=1e-6,
                    seed=17 + 1000 * k,  # semilla distinta por intento
                    verbose=fp_verbose,
                )

                # Si FP entrega solución factible, la guardamos y salimos
                if sol_dict is not None:
                    best_sol_vec = [sol_dict[v.VarName] for v in xvars]
                    break

            # Si hubo solución factible en presolve, se inyecta
            if best_sol_vec is not None:
                model._fp_inject.append(best_sol_vec)  # para cbSetSolution en nodo
                if inject_as_start:
                    # Opcional: setear MIP start (v.start)
                    for v, val in zip(xvars, best_sol_vec):
                        v.start = val
                if driver_verbose:
                    print(
                        "[Presolve] FP encontró solución factible -> inyectada como MIP start."
                    )
            else:
                if driver_verbose:
                    print("[Presolve] FP no encontró solución factible.")

        # -------------------- RONDAS DE B&B --------------------
        for r in range(1, rounds + 1):
            if driver_verbose:
                print(f"\n=== Ronda {r}/{rounds}: NodeLimit={nodes_per_round} ===")

            # Limpia el pool de relajaciones para esta ronda
            model._fp_pool = []

            # Fija el límite de nodos y ejecuta B&B con el callback
            model.Params.NodeLimit = nodes_per_round
            model.optimize(cb)

            # Recupera el pool recolectado por el callback
            pool = model._fp_pool
            if driver_verbose:
                print(f"[Ronda {r}] relajaciones recolectadas: {len(pool)}")

            # Si no hay semillas, pasa a la siguiente ronda
            if not pool:
                continue

            # Selecciona las mejores 'seeds_per_round' según cercanía a integridad
            seeds = sorted(pool, key=frac_distance)[: max(1, seeds_per_round)]

            best_sol_vec = (
                None  # reinicia “mejor” solución factible de FP en esta ronda
            )

            # Prueba FP sobre cada semilla seleccionada (primer factible gana)
            for k, x0 in enumerate(seeds, 1):
                if driver_verbose:
                    print(
                        f"[Ronda {r}] FP seed {k}/{len(seeds)}  ||frac||={frac_distance(x0):.3f}"
                    )

                sol_dict = feasibility_pump_seeded(
                    model,
                    xvars,
                    x0,
                    max_iters=fp_max_iters,
                    time_limit=fp_time_limit,
                    int_eps=1e-6,
                    seed=42 + 100 * r + k,  # semilla distinta por ronda/seed
                    verbose=fp_verbose,
                )

                if sol_dict is not None:
                    best_sol_vec = [sol_dict[v.VarName] for v in xvars]
                    break  # nos quedamos con la primera factible (puedes quitar el break si quieres)

            # Si FP encuentra solución, la encolamos para inyección y opcionalmente MIP start
            if best_sol_vec is not None:
                model._fp_inject.append(best_sol_vec)
                if inject_as_start:
                    for v, val in zip(xvars, best_sol_vec):
                        v.start = val
                if driver_verbose:
                    print(
                        f"[Ronda {r}] FP encontró solución factible -> encolada para inyección."
                    )
            else:
                if driver_verbose:
                    print(f"[Ronda {r}] FP no encontró solución factible.")

        # -------------------- RONDA FINAL SIN LÍMITE --------------------
        if driver_verbose:
            print("\n=== Ronda final: sin NodeLimit ===")
        model.Params.NodeLimit = GRB.INFINITY  # remueve límite de nodos
        model.optimize(cb)  # corrida final completa

    finally:
        # Siempre restaura parámetros del modelo, incluso si hubo excepciones
        if orig_NodeLimit is not None:
            model.Params.NodeLimit = orig_NodeLimit
        if orig_TimeLimit is not None:
            model.Params.TimeLimit = orig_TimeLimit

    # Devuelve el modelo optimizado
    return model

Y, ahora llamamos la instancia:

In [61]:
from helpers import instancia_knapsack


def crearModelo(nombre):
    m = gp.Model(nombre)

    params = {
        # === DESACTIVAR COSAS ===
        "Heuristics": 0.0,  # 0 desactiva heurísticas internas
        "Cuts": 0,  # 0 desactiva cortes
        "Presolve": 0,  # 0 desactiva presolve (2 agresivo, -1 auto)
        "Symmetry": 0,  # 0 desactiva detección de simetría
        "ConcurrentMIP": 1,  # 1 desactiva concurrente (corre estrategias distintas en paralelo)
        # === FOCO DE BÚSQUEDA ===
        "MIPFocus": 0,  # 0 auto, 1 incumbente rápido, 2 gap, 3 bound
        "VarBranch": 0,  # 0 auto, 1 max infeas, 2 pseudo cost; como se elige la variable actual para hacer branching
        "NodeMethod": 1,  # 1 dual simplex en nodos (0 auto, 2 barrier); como se resuelve el LP en los nodos
        "BranchDir": 0,  # 0 auto, 1 up, -1 down
        # === TOLERANCIAS ===
        "MIPGap": 0.0,  # gap objetivo relativo deseado (p.ej. 0.01 = 1%)
        "FeasibilityTol": 1e-6,  # tolerancia de viabilidad
        "IntFeasTol": 1e-5,  # tolerancia de integralidad
        "NumericFocus": 0,  # 0-3 (3 = más robusto numéricamente)
        # === LÍMITES Y LOG ===
        "TimeLimit": 60,  # seg. (0 = sin límite)
        "BestObjStop": None,  # para minimización: detiene al llegar a obj <= valor
        "BestBdStop": None,  # detiene si bound <= valor
        "Threads": 0,  # 0 = auto
        "Seed": 42,
        "LogToConsole": 1,  # 1 muestra log, 0 oculta
    }

    for k, v in params.items():
        if v is not None:
            m.setParam(k, v)
    return m


m = crearModelo("Auxiliar 8")

N = 1000
M = 5

v, W, caps = instancia_knapsack(
    n=N, m=M, density=0.26, seed=42, pesos_chicos=True
)  # density más baja = más difícil

x = m.addVars(N, vtype=GRB.BINARY, name="x")


# Objetivo: maximizar valor
m.setObjective(sum(v[i] * x[i] for i in range(N)), GRB.MAXIMIZE)

for j in range(len(caps)):
    m.addConstr(sum(W[j][i] * x[i] for i in range(N)) <= caps[j], name=f"cap_{j}")

# Atributos para callback
m._x = [x[i] for i in range(N)]
m._v = v
m._W = W
m._caps = caps
m._nodes_hist, m._ub_hist, m._lb_hist = (
    [],
    [],
    [],
)  # Las variables para graficar la respuesta

m.update()
m = run_bnb_with_fp_rounds(
    model=m,
    xvars=m._x,
    rounds=5,
    nodes_per_round=1000,
    seeds_per_round=500,
    fp_max_iters=40,
    fp_time_limit=10.0,
    inject_as_start=True,
    fp_verbose=True,
)

Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Set parameter Presolve to value 0
Set parameter Symmetry to value 0
Set parameter ConcurrentMIP to value 1
Set parameter MIPFocus to value 0
Set parameter VarBranch to value 0
Set parameter NodeMethod to value 1
Set parameter BranchDir to value 0
Set parameter MIPGap to value 0
Set parameter FeasibilityTol to value 1e-06
Set parameter IntFeasTol to value 1e-05
Set parameter NumericFocus to value 0
Set parameter TimeLimit to value 60
Set parameter Threads to value 0
Set parameter Seed to value 42
Set parameter LogToConsole to value 1

=== Presolve FP: usando relajación LP raíz ===
[Presolve] FP seed 1/3  ||frac||=0.812
[FP-seeded] it=0  ||x-z||_1(int)=0.811617
[FP-seeded] it=1  ||x-z||_1(int)=0
[FP-seeded] solución factible en it=1.
[FP-seeded] valor objetivo de la solución propuesta: 493.676
[Presolve] FP encontró solución factible -> inyectada como MIP start.

=== Ronda 1/5: NodeLimit=1000 ===
Set parameter NodeLimit to

Vemos que no nos fue muy bien encontrando soluciones óptimas, aunque el presolve estableció una buena solución de partida. Por lo mismo, propongamos algún método de mejora

## Mejoras al Feasibility Pump

Existen varios tipos de mejora, tanto para la fase de **proyección**, **redondeo** o **propagación**. 

### Temporary Fixing 

Consiste en que, en vez de redondear todas las variables de forma simultánea, mezclarlo con **propagación de restricciones**. Entonces, a la función `feasibility_pump_seeded` le creamos la función interna `sequential_round_with_propagation`

In [63]:
def feasibility_pump_seeded(
    model,  # Modelo MIP original (Gurobi)
    xvars,  # Lista ordenada de variables de decisión (mismo orden siempre)
    x0,  # Semilla inicial: solución LP (relajada) que viene del callback/B&B
    max_iters=40,  # Máx. iteraciones del bucle de Feasibility Pump
    time_limit=10.0,  # Límite de tiempo total (seg.) para este FP
    int_eps=1e-6,  # Tolerancia de “casi entero” para redondeo simple
    seed=0,  # Semilla aleatoria (perturbaciones, desempates)
    verbose=False,  # Si True, imprime trazas de progreso
    rounding_order="index",  # Orden de fijación: "index" | "fracdesc" | "fracasc"
    auto_fix_eps=1e-8,  # Si una entera queda a <= auto_fix_eps del entero, se fija automáticamente
    try_alt_round=True,  # Si al fijar al entero más cercano el LP se vuelve infactible, prueba floor/ceil
):  # cierra firma
    """
    Feasibility Pump con redondeo secuencial + propagación usando un LP de prueba ("probe").
    - Alterna: proyectar (resolver LP con capa |x - z|) y redondear (con propagación secuencial).
    - Usa un segundo LP (“probe”) para fijar enteras una a una y propagar restricciones.
    """

    t0 = time.time()  # Guarda tiempo de inicio (para respetar time_limit)
    random.seed(seed)  # Fija semilla de aleatoriedad para reproducibilidad

    vtypes = [
        v.VType for v in xvars
    ]  # Tipos Gurobi por variable (BINARY/INTEGER/CONTINUOUS)
    mask_int = [
        t in (GRB.BINARY, GRB.INTEGER) for t in vtypes
    ]  # Máscara booleana: True si variable es discreta

    # ---------- LP principal (proyección con norma L1) ----------
    lp, var_map = clone_as_lp(
        model, xvars
    )  # Clona el modelo como LP (todas las variables continuas)
    x_lp_vars = [var_map[v] for v in xvars]  # Variables clonadas alineadas con xvars
    # Añade capa de distancia: crea d_j y restricciones d_j >= |x_j - z_j|, y define objetivo min sum d_j
    d_vars, cons1, cons2 = build_distance_layer(lp, x_lp_vars, name_prefix="fp")

    # ---------- LP de prueba ("probe") para fijar + propagar ----------
    probe_lp, probe_map = clone_as_lp(
        model, xvars
    )  # Segundo LP independiente para probar fijaciones
    probe_vars = [probe_map[v] for v in xvars]  # Variables en el LP de prueba
    probe_orig_lb = [
        v.LB for v in probe_vars
    ]  # Guarda cotas inferiores originales (para restaurar)
    probe_orig_ub = [v.UB for v in probe_vars]  # Guarda cotas superiores originales

    def reset_probe_bounds():
        """Restaura las cotas originales del LP de prueba."""
        for pv, lb, ub in zip(probe_vars, probe_orig_lb, probe_orig_ub):
            pv.LB = lb  # Restaura LB
            pv.UB = ub  # Restaura UB
        probe_lp.update()  # Aplica cambios al modelo de prueba

    def round_nearest_int(val, vtype, lb, ub):
        """Redondea al entero más cercano respetando cotas; para binarias usa 0/1 con umbral 0.5."""
        if vtype == GRB.BINARY:  # Caso binario: umbral 0.5
            return 1 if val >= 0.5 else 0
        r = int(round(val))  # Entero más cercano
        if r < lb:
            r = int(lb)  # Respeta LB
        if r > ub:
            r = int(ub)  # Respeta UB
        return r

    def alt_int_values(val, vtype, lb, ub):
        """Devuelve alternativas floor/ceil distintas al entero más cercano (si existen y dentro de cotas)."""
        if vtype == GRB.BINARY:  # En binaria, la alternativa es el complemento
            return [1 - (1 if val >= 0.5 else 0)]
        floor_v = int(max(lb, min(ub, int(val // 1))))  # Floor dentro de [lb, ub]
        ceil_v = int(max(lb, min(ub, floor_v + 1)))  # Ceil dentro de [lb, ub]
        r = int(round(val))  # Redondeo principal
        alts = []  # Acumula alternativas válidas
        if floor_v != r:
            alts.append(floor_v)
        if ceil_v != r and ceil_v != floor_v:
            alts.append(ceil_v)
        alts = [
            a for a in dict.fromkeys(alts) if lb <= a <= ub
        ]  # Quita duplicados y re-chequea cotas
        return alts

    def sequential_round_with_propagation(x_start):
        """
        Fija enteras una a una en el LP de prueba y reoptimiza:
        - Orden de fijación según rounding_order.
        - Si fijación al entero más cercano infactibiliza, prueba alternativas (floor/ceil).
        - Tras cada reopt: auto-fija enteras “casi enteras” (<= auto_fix_eps).
        - Devuelve z_new (enteras fijadas) y deja continuas = x_start.
        """
        reset_probe_bounds()  # Restaura cotas del LP de prueba
        for j, pv in enumerate(
            probe_vars
        ):  # Asegura que todas estén libres inicialmente
            pv.LB = probe_orig_lb[j]
            pv.UB = probe_orig_ub[j]
        probe_lp.update()  # Aplica las restauraciones

        idxs = [
            j for j, is_int in enumerate(mask_int) if is_int
        ]  # Índices de variables enteras/binarias
        if rounding_order == "index":  # Orden natural de índices
            order = idxs
        else:
            fracs = [
                (j, abs(x_start[j] - round(x_start[j]))) for j in idxs
            ]  # Fraccionalidad |x - round(x)|
            rev = (
                rounding_order == "fracdesc"
            )  # Descendente si "fracdesc"; ascendente si "fracasc"
            fracs.sort(key=lambda t: t[1], reverse=rev)  # Ordena por fraccionalidad
            order = [j for j, _ in fracs]  # Solo los índices en el orden definido

        x_curr = list(x_start)  # Copia de valores actuales (por si se necesita)
        fixed = set()  # Conjunto de índices ya fijados

        def auto_fix_integers():
            """Fija automáticamente variables que queden “casi enteras” tras reoptimizar; reoptimiza en lazo."""
            changed = True
            while changed:
                changed = False
                x_vals = [
                    pv.X for pv in probe_vars
                ]  # Lee valores actuales del LP de prueba
                for k in idxs:
                    if k in fixed:  # Salta las ya fijadas
                        continue
                    vk = probe_vars[k]  # Variable a evaluar
                    r = round_nearest_int(
                        x_vals[k], vtypes[k], probe_orig_lb[k], probe_orig_ub[k]
                    )  # Entero destino
                    if (
                        abs(x_vals[k] - r) <= auto_fix_eps
                    ):  # Si quedó “pegada” a un entero…
                        vk.LB = r  # … la fijamos a r
                        vk.UB = r
                        fixed.add(k)  # Marcamos como fijada
                        changed = True  # Hubo cambio, reoptimizar de nuevo
                if changed:
                    probe_lp.optimize()  # Reoptimiza tras fijaciones automáticas
                    if probe_lp.status not in (GRB.OPTIMAL, GRB.SUBOPTIMAL):
                        return False  # Si se volvió infactible, aborta
            return True  # Terminó sin romper

        probe_lp.optimize()  # Resuelve estado base del LP de prueba
        if probe_lp.status not in (GRB.OPTIMAL, GRB.SUBOPTIMAL):
            return None  # Si ni siquiera hay base, aborta con None

        for j in order:  # Recorre variables en el orden decidido
            if j in fixed:  # Si ya se fijó por auto-fix, salta
                continue
            pv = probe_vars[j]  # Variable a fijar
            xj = (
                pv.X if pv.X is not None else x_curr[j]
            )  # Valor actual de la var (fallback a x_curr)
            r = round_nearest_int(
                xj, vtypes[j], probe_orig_lb[j], probe_orig_ub[j]
            )  # Entero más cercano

            pv.LB = r  # Intenta fijar j = r
            pv.UB = r
            probe_lp.update()
            probe_lp.optimize()  # Reoptimiza con esa fijación

            if probe_lp.status not in (GRB.OPTIMAL, GRB.SUBOPTIMAL):  # Si rompió…
                if try_alt_round:  # Opción: probar alternativas floor/ceil
                    ok = False
                    for alt in alt_int_values(
                        xj, vtypes[j], probe_orig_lb[j], probe_orig_ub[j]
                    ):
                        pv.LB = alt
                        pv.UB = alt
                        probe_lp.update()
                        probe_lp.optimize()
                        if probe_lp.status in (GRB.OPTIMAL, GRB.SUBOPTIMAL):
                            r = alt  # Acepta alternativa que funcione
                            ok = True
                            break
                    if not ok:  # Si ninguna alternativa funciona…
                        pv.LB = probe_orig_lb[j]  # … suelta j
                        pv.UB = probe_orig_ub[j]
                        probe_lp.update()
                        probe_lp.optimize()
                        continue  # y continúa con la siguiente variable
                else:
                    pv.LB = probe_orig_lb[j]  # Si no se permiten alternativas, suelta j
                    pv.UB = probe_orig_ub[j]
                    probe_lp.update()
                    probe_lp.optimize()
                    continue

            fixed.add(j)  # Marca j como fijada satisfactoriamente

            ok = auto_fix_integers()  # Propaga: fija otras casi enteras y reoptimiza
            if not ok:  # Si la propagación rompió…
                pv.LB = probe_orig_lb[j]  # … deshaz la última fijación j
                pv.UB = probe_orig_ub[j]
                fixed.discard(j)
                probe_lp.update()
                probe_lp.optimize()
                # No reintenta aquí (las alternativas ya se probaron arriba)

        if probe_lp.status not in (GRB.OPTIMAL, GRB.SUBOPTIMAL):
            return None  # Seguridad: si quedó infactible, aborta

        x_final = [pv.X for pv in probe_vars]  # Lee la solución final del LP de prueba
        z_out = []  # Construirá el vector de enteras final
        for k, is_int in enumerate(mask_int):
            if is_int:  # Para enteras/binarias: entero consistente con cotas
                z_out.append(
                    round_nearest_int(
                        x_final[k], vtypes[k], probe_orig_lb[k], probe_orig_ub[k]
                    )
                )
            else:
                z_out.append(x_start[k])  # Para continuas: mantenemos el valor base
        return z_out  # Devuelve z_new redondeado con propagación

    # ---------- z inicial: redondeo de la semilla x0 ----------
    z = [
        (
            round_nearest_int(x0[j], vtypes[j], xvars[j].LB, xvars[j].UB)
            if mask_int[j]
            else x0[j]
        )
        for j in range(len(xvars))
    ]
    best_L1 = distance_L1(x0, z, mask_int)  # Distancia L1 inicial (solo sobre enteras)
    if verbose:
        print(f"[FP-seeded] it=0  ||x-z||_1(int)={best_L1:.6g}")

    update_z_rhs(lp, cons1, cons2, z)  # Sincroniza RHS de la capa de distancia con z

    z_history = set()  # Conjunto de huellas (anticiclo)
    z_key = tuple(
        int(zj) if mask_int[j] else 0 for j, zj in enumerate(z)
    )  # Huella: solo enteras
    z_history.add(z_key)  # Registra la primera huella

    it, no_improve = 0, 0  # Contadores de iteración y estancamiento
    while (
        it < max_iters and (time.time() - t0) < time_limit
    ):  # Criterio de parada por iteraciones/tiempo
        it += 1  # Avanza iteración

        lp.optimize()  # Proyección: resuelve min sum d_j (||x - z||_1)
        if lp.status not in (GRB.OPTIMAL, GRB.SUBOPTIMAL):  # Si LP falló, aborta
            if verbose:
                print(f"[FP-seeded] LP infactible en it={it}.")
            break

        x_rel = [
            xj.X for xj in x_lp_vars
        ]  # Solución relajada x (continua) tras proyección

        # Redondeo avanzado: usa el LP de prueba para fijar secuencialmente + propagar
        z_new = sequential_round_with_propagation(x_rel)
        if z_new is None:  # Fallback: redondeo simple si algo falló
            z_new = [
                (
                    round_nearest_int(x_rel[j], vtypes[j], xvars[j].LB, xvars[j].UB)
                    if mask_int[j]
                    else x_rel[j]
                )
                for j in range(len(x_rel))
            ]

        L1 = distance_L1(x_rel, z_new, mask_int)  # Distancia L1 tras redondeo
        if verbose:
            print(f"[FP-seeded] it={it}  ||x-z||_1(int)={L1:.6g}")

        if L1 + 1e-12 < best_L1:  # Mejora estricta -> resetea estancamiento
            best_L1 = L1
            no_improve = 0
        else:
            no_improve += 1  # Si no mejora, acumula estancamiento

        # Chequeo: ¿x_rel ya es (casi) entera en las componentes discretas?
        if all(
            (not mask_int[j]) or abs(x_rel[j] - round(x_rel[j])) <= 1e-8
            for j in range(len(x_rel))
        ):
            # Intenta construir solución factible fijando enteras y resolviendo modelo original
            ok, sol = check_feasible_by_fixing_integers(
                model,
                xvars,
                [
                    round(x_rel[j]) if mask_int[j] else x_rel[j]
                    for j in range(len(x_rel))
                ],
            )
            if ok:  # Si factible: reporta y devuelve
                if verbose:
                    print(f"[FP-seeded] solución factible en it={it}.")
                    if hasattr(
                        model, "_v"
                    ):  # (Opcional) calcula objetivo si hay vector de costos
                        obj_val = sum(
                            model._v[i] * sol[f"x[{i}]"] for i in range(len(model._v))
                        )
                        print(
                            f"[FP-seeded] valor objetivo de la solución propuesta: {obj_val:.3f}"
                        )
                        if model.SolCount > 0:
                            best = model.ObjVal
                            print(f"[FP-seeded] incumbente actual: {best:.3f}")
                            print(
                                f"[FP-seeded] {'MEJORA!' if obj_val > best else 'no mejora.'}"
                            )
                return sol  # Éxito: se devuelve diccionario VarName -> valor

        # Anticiclo / perturbación: si se repite z o hay 3 iteraciones sin mejora
        z_key = tuple(
            int(z_new[j]) if mask_int[j] else 0 for j in range(len(z_new))
        )  # Huella de z_new
        if (z_key in z_history) or (no_improve >= 3):
            # Ordena por fraccionalidad para decidir qué variables alterar
            fracs = sorted(
                [
                    (j, abs(x_rel[j] - round(x_rel[j])))
                    for j in range(len(x_rel))
                    if mask_int[j]
                ],
                key=lambda t: t[1],
                reverse=True,
            )
            frac_order = [j for j, _ in fracs]  # Índices de más a menos fraccionales
            flips = 2 if len(frac_order) >= 2 else 1  # Nº de flips a aplicar
            if verbose:
                print(f"[FP-seeded] perturbación (flips={flips}) en it={it}.")
            # Aplica pequeña perturbación en z_new para escapar del ciclo/estancamiento
            z_new = perturb_z(
                z_new,
                vtypes,
                num_flips=flips,
                frac_order=frac_order,
                seed=seed + 1234 + it,
            )
            no_improve = 0  # Resetea contador de no-mejora

        z_history.add(z_key)  # Registra huella para detección de ciclos
        z = z_new  # Actualiza z para la próxima iteración
        update_z_rhs(
            lp, cons1, cons2, z
        )  # Actualiza RHS de capa |x - z| con el nuevo z

    if verbose:
        print(
            "[FP-seeded] sin solución factible en límites dados."
        )  # Mensaje final si no hubo éxito
    return None  # No se encontró solución dentro de límites

Ahora, volvamos a correr bajo el mismo ejemplo de antes.

In [14]:
def run_bnb_with_fp_rounds(
    model,
    xvars,
    rounds=5,
    nodes_per_round=5000,
    collect_every_k_nodes=1,
    pool_cap=5000,
    seeds_per_round=3,
    fp_max_iters=40,
    fp_time_limit=10.0,
    inject_as_start=True,
    fp_verbose=False,
    driver_verbose=True,
    presolve_fp=True,
    presolve_seeds=3,  # cuántas semillas probar en presolve
    presolve_try_round_pert=True,  # añadir algunas perturbaceanos
    presolve_time_limit=None,  #
):
    """
    - Primero (opcional): corre un Feasibility Pump 'presolve' seed-eado desde la relajación raíz.
      Si encuentra factible, la inyecta (MIP start + cola).
    - Luego ejecuta B&B en 'rounds' (NodeLimit por ronda).
    - Entre rondas: FP seed-eado con semillas del pool recolectado por callback.
    - Encola soluciones enteras para que el callback las inyecte en la siguiente ronda.
    - Opcionalmente setea 'v.start' con la mejor solución factible encontrada por FP.
    """
    # callback (usa _fp_pool y _fp_inject)
    cb = make_feaspump_rounds_callback(
        model, xvars, collect_every_k_nodes=collect_every_k_nodes, pool_cap=pool_cap
    )

    # asegurar estructuras
    if not hasattr(model, "_fp_inject"):
        model._fp_inject = []
    if not hasattr(model, "_fp_pool"):
        model._fp_pool = []

    # guardar params originales a restaurar
    orig_NodeLimit = model.Params.NodeLimit
    orig_TimeLimit = model.Params.TimeLimit

    # helper: cercanía a enteros
    vtypes = [v.VType for v in xvars]
    mask_int = [t in (GRB.BINARY, GRB.INTEGER) for t in vtypes]

    def frac_distance(x):
        return sum(abs(x[j] - round(x[j])) for j in range(len(x)) if mask_int[j])

    # ====== helper interno: obtener relajación raíz y semillas ======
    def _root_relax_and_seeds():
        # Clonar como LP y resolver
        lp_relax, var_map = clone_as_lp(model, xvars)
        x_lp_vars = [var_map[v] for v in xvars]
        lp_relax.optimize()
        if lp_relax.status not in (GRB.OPTIMAL, GRB.SUBOPTIMAL):
            return []

        x0 = [v.X for v in x_lp_vars]

        seeds = [x0]
        if presolve_try_round_pert and presolve_seeds > 1:
            # Semillas extras: pequeñas perturbaciones dirigidas por fraccionalidad
            fracs = [
                (j, abs(x0[j] - round(x0[j]))) for j in range(len(x0)) if mask_int[j]
            ]
            fracs.sort(key=lambda t: t[1], reverse=True)
            top_idx = [j for j, _ in fracs[: max(3, min(10, len(fracs)))]]
            rng = random.Random(12345)
            for s in range(1, presolve_seeds):
                xd = x0[:]
                # voltear/fijar algunos de los más fraccionales como seed distinta
                for j in top_idx:
                    if vtypes[j] == GRB.BINARY:
                        # flip con baja probabilidad para explorar
                        if rng.random() < 0.35:
                            xd[j] = 1.0 - round(x0[j])
                        else:
                            xd[j] = round(x0[j])
                    else:
                        # enteras generales: empujar hacia floor/ceil cercano
                        r = round(x0[j])
                        if rng.random() < 0.5:
                            xd[j] = r
                        else:
                            # pequeña variación (mantener en [LB,UB] si tienes acceso)
                            xd[j] = r
                seeds.append(xd)
        return seeds

    try:
        # ==================== PRESOLVE FP ====================
        if presolve_fp:
            if driver_verbose:
                print("\n=== Presolve FP: usando relajación LP raíz ===")
            presolve_seeds = max(1, presolve_seeds)
            root_seeds = _root_relax_and_seeds()

            best_sol_vec = None
            for k, x0 in enumerate(root_seeds[:presolve_seeds], 1):
                if driver_verbose:
                    print(
                        f"[Presolve] FP seed {k}/{min(presolve_seeds,len(root_seeds))}  ||frac||={frac_distance(x0):.3f}"
                    )
                sol_dict = feasibility_pump_seeded(
                    model,
                    xvars,
                    x0,
                    max_iters=fp_max_iters,
                    time_limit=(
                        presolve_time_limit
                        if presolve_time_limit is not None
                        else fp_time_limit
                    ),
                    int_eps=1e-6,
                    seed=17 + 1000 * k,
                    verbose=fp_verbose,
                )
                if sol_dict is not None:
                    best_sol_vec = [sol_dict[v.VarName] for v in xvars]
                    break

            if best_sol_vec is not None:
                # inyectar como start y para el callback
                model._fp_inject.append(best_sol_vec)
                if inject_as_start:
                    for v, val in zip(xvars, best_sol_vec):
                        v.start = val
                if driver_verbose:
                    print(
                        "[Presolve] FP encontró solución factible -> inyectada como MIP start."
                    )
            else:
                if driver_verbose:
                    print("[Presolve] FP no encontró solución factible.")

        # ==================== RONDAS B&B ====================
        for r in range(1, rounds + 1):
            if driver_verbose:
                print(f"\n=== Ronda {r}/{rounds}: NodeLimit={nodes_per_round} ===")

            # limpiar pool para la ronda
            model._fp_pool = []

            # fijar NodeLimit para esta ronda y correr
            model.Params.NodeLimit = nodes_per_round
            model.optimize(cb)

            pool = model._fp_pool
            if driver_verbose:
                print(f"[Ronda {r}] relajaciones recolectadas: {len(pool)}")

            if not pool:
                continue

            # elegir semillas (más cercanas a enteros)
            seeds = sorted(pool, key=frac_distance)[: max(1, seeds_per_round)]

            best_sol_vec = None
            for k, x0 in enumerate(seeds, 1):
                if driver_verbose:
                    print(
                        f"[Ronda {r}] FP seed {k}/{len(seeds)}  ||frac||={frac_distance(x0):.3f}"
                    )
                sol_dict = feasibility_pump_seeded(
                    model,
                    xvars,
                    x0,
                    max_iters=fp_max_iters,
                    time_limit=fp_time_limit,
                    int_eps=1e-6,
                    seed=42 + 100 * r + k,
                    verbose=fp_verbose,
                )
                if sol_dict is not None:
                    best_sol_vec = [sol_dict[v.VarName] for v in xvars]
                    break  # nos quedamos con la primera factible (puedes quitar el break si quieres)

            if best_sol_vec is not None:
                # encolar para inyección en la próxima ronda
                model._fp_inject.append(best_sol_vec)
                if inject_as_start:
                    for v, val in zip(xvars, best_sol_vec):
                        v.start = val
                if driver_verbose:
                    print(
                        f"[Ronda {r}] FP encontró solución factible -> encolada para inyección."
                    )
            else:
                if driver_verbose:
                    print(f"[Ronda {r}] FP no encontró solución factible.")

        # ==================== RONDA FINAL ====================
        if driver_verbose:
            print("\n=== Ronda final: sin NodeLimit ===")
        model.Params.NodeLimit = GRB.INFINITY
        model.optimize(cb)

    finally:
        # restaurar
        if orig_NodeLimit is not None:
            model.Params.NodeLimit = orig_NodeLimit
        if orig_TimeLimit is not None:
            model.Params.TimeLimit = orig_TimeLimit

    return m

In [64]:
from helpers import instancia_knapsack


def crearModelo(nombre):
    m = gp.Model(nombre)

    params = {
        # === DESACTIVAR COSAS ===
        "Heuristics": 0.0,  # 0 desactiva heurísticas internas
        "Cuts": 0,  # 0 desactiva cortes
        "Presolve": 0,  # 0 desactiva presolve (2 agresivo, -1 auto)
        "Symmetry": 0,  # 0 desactiva detección de simetría
        "ConcurrentMIP": 1,  # 1 desactiva concurrente (corre estrategias distintas en paralelo)
        # === FOCO DE BÚSQUEDA ===
        "MIPFocus": 0,  # 0 auto, 1 incumbente rápido, 2 gap, 3 bound
        "VarBranch": 0,  # 0 auto, 1 max infeas, 2 pseudo cost; como se elige la variable actual para hacer branching
        "NodeMethod": 1,  # 1 dual simplex en nodos (0 auto, 2 barrier); como se resuelve el LP en los nodos
        "BranchDir": 0,  # 0 auto, 1 up, -1 down
        # === TOLERANCIAS ===
        "MIPGap": 0.0,  # gap objetivo relativo deseado (p.ej. 0.01 = 1%)
        "FeasibilityTol": 1e-6,  # tolerancia de viabilidad
        "IntFeasTol": 1e-5,  # tolerancia de integralidad
        "NumericFocus": 0,  # 0-3 (3 = más robusto numéricamente)
        # === LÍMITES Y LOG ===
        "TimeLimit": 60,  # seg. (0 = sin límite)
        "BestObjStop": None,  # para minimización: detiene al llegar a obj <= valor
        "BestBdStop": None,  # detiene si bound <= valor
        "Threads": 0,  # 0 = auto
        "Seed": 42,
        "LogToConsole": 1,  # 1 muestra log, 0 oculta
    }

    for k, v in params.items():
        if v is not None:
            m.setParam(k, v)
    return m


m = crearModelo("Auxiliar 8")

N = 1000
M = 2

v, W, caps = instancia_knapsack(
    n=N, m=M, density=0.56, seed=42, pesos_chicos=True
)  # density más baja = más difícil

x = m.addVars(N, vtype=GRB.BINARY, name="x")


# Objetivo: maximizar valor
m.setObjective(sum(v[i] * x[i] for i in range(N)), GRB.MAXIMIZE)

for j in range(len(caps)):
    m.addConstr(sum(W[j][i] * x[i] for i in range(N)) <= caps[j], name=f"cap_{j}")

# Atributos para callback
m._x = [x[i] for i in range(N)]
m._v = v
m._W = W
m._caps = caps
m._nodes_hist, m._ub_hist, m._lb_hist = (
    [],
    [],
    [],
)  # Las variables para graficar la respuesta

m.update()
m = run_bnb_with_fp_rounds(
    model=m,
    xvars=m._x,
    rounds=5,
    nodes_per_round=500,
    collect_every_k_nodes=5,
    pool_cap=3000,
    seeds_per_round=100,
    fp_max_iters=40,
    fp_time_limit=10.0,
    inject_as_start=True,
    fp_verbose=True,
    driver_verbose=True,
)

Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Set parameter Presolve to value 0
Set parameter Symmetry to value 0
Set parameter ConcurrentMIP to value 1
Set parameter MIPFocus to value 0
Set parameter VarBranch to value 0
Set parameter NodeMethod to value 1
Set parameter BranchDir to value 0
Set parameter MIPGap to value 0
Set parameter FeasibilityTol to value 1e-06
Set parameter IntFeasTol to value 1e-05
Set parameter NumericFocus to value 0
Set parameter TimeLimit to value 60
Set parameter Threads to value 0
Set parameter Seed to value 42
Set parameter LogToConsole to value 1

=== Presolve FP: usando relajación LP raíz ===
[Presolve] FP seed 1/3  ||frac||=0.600
[FP-seeded] it=0  ||x-z||_1(int)=0.6
[FP-seeded] it=1  ||x-z||_1(int)=1.6
[FP-seeded] it=2  ||x-z||_1(int)=0
[FP-seeded] solución factible en it=2.
[FP-seeded] valor objetivo de la solución propuesta: 748.965
[Presolve] FP encontró solución factible -> inyectada como MIP start.

=== Ronda 1/5: NodeLimit=500

## Proyección 

Una desventaja de la proyección es que **no toma en cuenta la función objetivo**. Por lo mismo, es posible añadir una combinación convexa de la función objetivo:

$$c^Tx\leq\beta c^T\bar{x}+(1-\beta)c^T\tilde{x}$$

Con $\beta\in (0,1)$. Otra opción, más utilizada, es cambiar el objetivo de la proyección:

$$\Delta_\alpha (x,\tilde{x}):=(1-\alpha)\Delta(x,\tilde{x})+\alpha\frac{\sqrt{\lvert\mathcal{I}\rvert}}{\lvert\lvert c\rvert\rvert}c^Tx$$ 

Nos definiremos helpers nuevos, por lo tanto: 

In [16]:
def get_obj_coeffs_for(model, xvars):
    """Devuelve c_j = coef. lineal de la FO original para cada x_j."""
    name_to_var = {v.VarName: v for v in model.getVars()}
    return [name_to_var[v.VarName].Obj for v in xvars]


def add_obj_combo_constr(lp, x_lp_vars, c, sense, beta, cTx_bar, z, name="fp_obj_cap"):
    """
    Crea la restricción de 'proyección con objetivo':
      - Si MINIMIZE:   c^T x <= beta*c^T xbar + (1-beta)*c^T z
      - Si MAXIMIZE:   c^T x >= beta*c^T xbar + (1-beta)*c^T z
    Devuelve el objeto de restricción creado.
    """
    cTx_tilde = sum(c[j] * z[j] for j in range(len(z)))
    rhs = beta * cTx_bar + (1.0 - beta) * cTx_tilde
    expr = gp.quicksum(c[j] * x_lp_vars[j] for j in range(len(x_lp_vars)))
    if sense == GRB.MINIMIZE:
        cons = lp.addConstr(expr <= rhs, name=name)
    else:  # MAXIMIZE
        cons = lp.addConstr(expr >= rhs, name=name)
    lp.update()
    return cons


def update_obj_combo_rhs(lp, cons, c, sense, beta, cTx_bar, z):
    """Actualiza sólo el RHS con el nuevo \tilde{x}=z."""
    if cTx_bar is None:
        return  # nada que actualizar si no hay incumbente
    cTx_tilde = sum(c[j] * z[j] for j in range(len(z)))
    rhs = beta * cTx_bar + (1.0 - beta) * cTx_tilde
    cons.RHS = rhs
    lp.update()

Luego, nos volvemos a crear `feasibility_pump_seeded` con un parámetro `beta` que nos dirá cuanto ponderar en la combinación convexa

In [33]:
def feasibility_pump_seeded(
    model,
    xvars,
    x0,
    max_iters=40,
    time_limit=10.0,
    int_eps=1e-6,
    seed=0,
    verbose=False,
    beta=None,
):
    t0 = time.time()
    random.seed(seed)
    print("Corriendo feasibility pump con combinación convexa, con beta ", beta)

    vtypes = [v.VType for v in xvars]
    mask_int = [t in (GRB.BINARY, GRB.INTEGER) for t in vtypes]

    # LP auxiliar + capa distancia
    lp, var_map = clone_as_lp(model, xvars)
    x_lp_vars = [var_map[v] for v in xvars]
    d_vars, cons1, cons2 = build_distance_layer(lp, x_lp_vars, name_prefix="fp")

    # -------- NUEVO: preparar cap de objetivo (si corresponde) --------
    obj_cap_cons = None
    if beta is not None:
        # coeficientes c de la FO original
        c = get_obj_coeffs_for(model, xvars)
        sense = model.ModelSense  # GRB.MINIMIZE(1) o MAXIMIZE(-1)
        # c^T \bar{x}: si hay incumbente
        cTx_bar = None
        try:
            if model.SolCount > 0:
                cTx_bar = model.ObjVal  # valor objetivo de la incumbente
        except gp.GurobiError:
            cTx_bar = None
    # ------------------------------------------------------------------

    # z inicial
    z = round_ints(x0, vtypes, eps=int_eps, seed=seed)
    best_L1 = distance_L1(x0, z, mask_int)
    if verbose:
        print(f"[FP-seeded] it=0  ||x-z||_1(int)={best_L1:.6g}")

    # RHS para |x-z|
    update_z_rhs(lp, cons1, cons2, z)

    # -------- NUEVO: crear restricción de objetivo si procede ---------
    if beta is not None and cTx_bar is not None:
        obj_cap_cons = add_obj_combo_constr(
            lp, x_lp_vars, c, sense, beta, cTx_bar, z, name="fp_obj_cap"
        )
    # ------------------------------------------------------------------

    z_history = set()
    z_key = tuple(int(zj) if mask_int[j] else 0 for j, zj in enumerate(z))
    z_history.add(z_key)

    it, no_improve = 0, 0
    while it < max_iters and (time.time() - t0) < time_limit:
        it += 1
        lp.optimize()
        if lp.status not in (GRB.OPTIMAL, GRB.SUBOPTIMAL):
            if verbose:
                print(f"[FP-seeded] LP infactible en it={it}.")
            break

        x_rel = [xj.X for xj in x_lp_vars]
        z_new = round_ints(x_rel, vtypes, eps=int_eps, seed=seed + it)
        L1 = distance_L1(x_rel, z_new, mask_int)
        if verbose:
            print(f"[FP-seeded] it={it}  ||x-z||_1(int)={L1:.6g}")

        if L1 + 1e-12 < best_L1:
            best_L1, no_improve = L1, 0
        else:
            no_improve += 1

        # ¿x_rel entero?
        if all(
            (not mask_int[j]) or abs(x_rel[j] - round(x_rel[j])) <= 1e-8
            for j in range(len(x_rel))
        ):
            ok, sol = check_feasible_by_fixing_integers(
                model,
                xvars,
                [
                    round(x_rel[j]) if mask_int[j] else x_rel[j]
                    for j in range(len(x_rel))
                ],
            )
            if ok:
                if verbose:
                    print(f"[FP-seeded] solución factible en it={it}.")
                return sol

        # anticiclo
        z_key = tuple(int(z_new[j]) if mask_int[j] else 0 for j in range(len(z_new)))
        if (z_key in z_history) or (no_improve >= 3):
            fracs = sorted(
                [
                    (j, abs(x_rel[j] - round(x_rel[j])))
                    for j in range(len(x_rel))
                    if mask_int[j]
                ],
                key=lambda t: t[1],
                reverse=True,
            )
            frac_order = [j for j, _ in fracs]
            flips = 2 if len(frac_order) >= 2 else 1
            if verbose:
                print(f"[FP-seeded] perturbación (flips={flips}) en it={it}.")
            z_new = perturb_z(
                z_new,
                vtypes,
                num_flips=flips,
                frac_order=frac_order,
                seed=seed + 1234 + it,
            )
            no_improve = 0

        z_history.add(z_key)
        z = z_new

        # actualizar |x-z|
        update_z_rhs(lp, cons1, cons2, z)

        # -------- NUEVO: actualizar el RHS del cap --------
        if obj_cap_cons is not None:
            update_obj_combo_rhs(lp, obj_cap_cons, c, sense, beta, cTx_bar, z)
        # --------------------------------------------------

    if verbose:
        print("[FP-seeded] sin solución factible en límites dados.")
    return None

Y, nos volvemos a definir nuestras funciones antiguas:

In [35]:
def run_bnb_with_fp_rounds(
    model,
    xvars,
    rounds=5,
    nodes_per_round=5000,
    collect_every_k_nodes=1,
    pool_cap=5000,
    seeds_per_round=3,
    fp_max_iters=40,
    fp_time_limit=10.0,
    inject_as_start=True,
    fp_verbose=False,
    driver_verbose=True,
    presolve_fp=True,
    presolve_seeds=3,  # cuántas semillas probar en presolve
    presolve_try_round_pert=True,  # añadir algunas perturbaceanos
    presolve_time_limit=None,
    beta=0.7,  #
):
    """
    - Primero (opcional): corre un Feasibility Pump 'presolve' seed-eado desde la relajación raíz.
      Si encuentra factible, la inyecta (MIP start + cola).
    - Luego ejecuta B&B en 'rounds' (NodeLimit por ronda).
    - Entre rondas: FP seed-eado con semillas del pool recolectado por callback.
    - Encola soluciones enteras para que el callback las inyecte en la siguiente ronda.
    - Opcionalmente setea 'v.start' con la mejor solución factible encontrada por FP.
    """
    # callback (usa _fp_pool y _fp_inject)
    cb = make_feaspump_rounds_callback(
        model, xvars, collect_every_k_nodes=collect_every_k_nodes, pool_cap=pool_cap
    )

    # asegurar estructuras
    if not hasattr(model, "_fp_inject"):
        model._fp_inject = []
    if not hasattr(model, "_fp_pool"):
        model._fp_pool = []

    # guardar params originales a restaurar
    orig_NodeLimit = model.Params.NodeLimit
    orig_TimeLimit = model.Params.TimeLimit

    # helper: cercanía a enteros
    vtypes = [v.VType for v in xvars]
    mask_int = [t in (GRB.BINARY, GRB.INTEGER) for t in vtypes]

    def frac_distance(x):
        return sum(abs(x[j] - round(x[j])) for j in range(len(x)) if mask_int[j])

    # ====== helper interno: obtener relajación raíz y semillas ======
    def _root_relax_and_seeds():
        # Clonar como LP y resolver
        lp_relax, var_map = clone_as_lp(model, xvars)
        x_lp_vars = [var_map[v] for v in xvars]
        lp_relax.optimize()
        if lp_relax.status not in (GRB.OPTIMAL, GRB.SUBOPTIMAL):
            return []

        x0 = [v.X for v in x_lp_vars]

        seeds = [x0]
        if presolve_try_round_pert and presolve_seeds > 1:
            # Semillas extras: pequeñas perturbaciones dirigidas por fraccionalidad
            fracs = [
                (j, abs(x0[j] - round(x0[j]))) for j in range(len(x0)) if mask_int[j]
            ]
            fracs.sort(key=lambda t: t[1], reverse=True)
            top_idx = [j for j, _ in fracs[: max(3, min(10, len(fracs)))]]
            rng = random.Random(12345)
            for s in range(1, presolve_seeds):
                xd = x0[:]
                # voltear/fijar algunos de los más fraccionales como seed distinta
                for j in top_idx:
                    if vtypes[j] == GRB.BINARY:
                        # flip con baja probabilidad para explorar
                        if rng.random() < 0.35:
                            xd[j] = 1.0 - round(x0[j])
                        else:
                            xd[j] = round(x0[j])
                    else:
                        # enteras generales: empujar hacia floor/ceil cercano
                        r = round(x0[j])
                        if rng.random() < 0.5:
                            xd[j] = r
                        else:
                            # pequeña variación (mantener en [LB,UB] si tienes acceso)
                            xd[j] = r
                seeds.append(xd)
        return seeds

    try:
        # ==================== PRESOLVE FP ====================
        if presolve_fp:
            if driver_verbose:
                print("\n=== Presolve FP: usando relajación LP raíz ===")
            presolve_seeds = max(1, presolve_seeds)
            root_seeds = _root_relax_and_seeds()

            best_sol_vec = None
            for k, x0 in enumerate(root_seeds[:presolve_seeds], 1):
                if driver_verbose:
                    print(
                        f"[Presolve] FP seed {k}/{min(presolve_seeds,len(root_seeds))}  ||frac||={frac_distance(x0):.3f}"
                    )
                sol_dict = feasibility_pump_seeded(
                    model,
                    xvars,
                    x0,
                    max_iters=fp_max_iters,
                    time_limit=(
                        presolve_time_limit
                        if presolve_time_limit is not None
                        else fp_time_limit
                    ),
                    int_eps=1e-6,
                    seed=17 + 1000 * k,
                    beta=beta,
                    verbose=fp_verbose,
                )
                if sol_dict is not None:
                    best_sol_vec = [sol_dict[v.VarName] for v in xvars]
                    break

            if best_sol_vec is not None:
                # inyectar como start y para el callback
                model._fp_inject.append(best_sol_vec)
                if inject_as_start:
                    for v, val in zip(xvars, best_sol_vec):
                        v.start = val
                if driver_verbose:
                    print(
                        "[Presolve] FP encontró solución factible -> inyectada como MIP start."
                    )
            else:
                if driver_verbose:
                    print("[Presolve] FP no encontró solución factible.")

        # ==================== RONDAS B&B ====================
        for r in range(1, rounds + 1):
            if driver_verbose:
                print(f"\n=== Ronda {r}/{rounds}: NodeLimit={nodes_per_round} ===")

            # limpiar pool para la ronda
            model._fp_pool = []

            # fijar NodeLimit para esta ronda y correr
            model.Params.NodeLimit = nodes_per_round
            model.optimize(cb)

            pool = model._fp_pool
            if driver_verbose:
                print(f"[Ronda {r}] relajaciones recolectadas: {len(pool)}")

            if not pool:
                continue

            # elegir semillas (más cercanas a enteros)
            seeds = sorted(pool, key=frac_distance)[: max(1, seeds_per_round)]

            best_sol_vec = None
            for k, x0 in enumerate(seeds, 1):
                if driver_verbose:
                    print(
                        f"[Ronda {r}] FP seed {k}/{len(seeds)}  ||frac||={frac_distance(x0):.3f}"
                    )
                sol_dict = feasibility_pump_seeded(
                    model,
                    xvars,
                    x0,
                    max_iters=fp_max_iters,
                    time_limit=fp_time_limit,
                    int_eps=1e-6,
                    seed=42 + 100 * r + k,
                    verbose=fp_verbose,
                )
                if sol_dict is not None:
                    best_sol_vec = [sol_dict[v.VarName] for v in xvars]
                    break  # nos quedamos con la primera factible (puedes quitar el break si quieres)

            if best_sol_vec is not None:
                # encolar para inyección en la próxima ronda
                model._fp_inject.append(best_sol_vec)
                if inject_as_start:
                    for v, val in zip(xvars, best_sol_vec):
                        v.start = val
                if driver_verbose:
                    print(
                        f"[Ronda {r}] FP encontró solución factible -> encolada para inyección."
                    )
            else:
                if driver_verbose:
                    print(f"[Ronda {r}] FP no encontró solución factible.")

        # ==================== RONDA FINAL ====================
        if driver_verbose:
            print("\n=== Ronda final: sin NodeLimit ===")
        model.Params.NodeLimit = GRB.INFINITY
        model.optimize(cb)

    finally:
        # restaurar
        if orig_NodeLimit is not None:
            model.Params.NodeLimit = orig_NodeLimit
        if orig_TimeLimit is not None:
            model.Params.TimeLimit = orig_TimeLimit

    return m

In [38]:
from helpers import instancia_knapsack


def crearModelo(nombre):
    m = gp.Model(nombre)

    params = {
        # === DESACTIVAR COSAS ===
        "Heuristics": 0.0,  # 0 desactiva heurísticas internas
        "Cuts": 0,  # 0 desactiva cortes
        "Presolve": 0,  # 0 desactiva presolve (2 agresivo, -1 auto)
        "Symmetry": 0,  # 0 desactiva detección de simetría
        "ConcurrentMIP": 1,  # 1 desactiva concurrente (corre estrategias distintas en paralelo)
        # === FOCO DE BÚSQUEDA ===
        "MIPFocus": 0,  # 0 auto, 1 incumbente rápido, 2 gap, 3 bound
        "VarBranch": 0,  # 0 auto, 1 max infeas, 2 pseudo cost; como se elige la variable actual para hacer branching
        "NodeMethod": 1,  # 1 dual simplex en nodos (0 auto, 2 barrier); como se resuelve el LP en los nodos
        "BranchDir": 0,  # 0 auto, 1 up, -1 down
        # === TOLERANCIAS ===
        "MIPGap": 0.0,  # gap objetivo relativo deseado (p.ej. 0.01 = 1%)
        "FeasibilityTol": 1e-6,  # tolerancia de viabilidad
        "IntFeasTol": 1e-5,  # tolerancia de integralidad
        "NumericFocus": 0,  # 0-3 (3 = más robusto numéricamente)
        # === LÍMITES Y LOG ===
        "TimeLimit": 60,  # seg. (0 = sin límite)
        "BestObjStop": None,  # para minimización: detiene al llegar a obj <= valor
        "BestBdStop": None,  # detiene si bound <= valor
        "Threads": 0,  # 0 = auto
        "Seed": 42,
        "LogToConsole": 1,  # 1 muestra log, 0 oculta
    }

    for k, v in params.items():
        if v is not None:
            m.setParam(k, v)
    return m


m = crearModelo("Auxiliar 7")

N = 1000
M = 2

v, W, caps = instancia_knapsack(
    n=N, m=M, density=0.56, seed=42, pesos_chicos=True
)  # density más baja = más difícil

x = m.addVars(N, vtype=GRB.BINARY, name="x")


# Objetivo: maximizar valor
m.setObjective(sum(v[i] * x[i] for i in range(N)), GRB.MAXIMIZE)

for j in range(len(caps)):
    m.addConstr(sum(W[j][i] * x[i] for i in range(N)) <= caps[j], name=f"cap_{j}")

# Atributos para callback
m._x = [x[i] for i in range(N)]
m._v = v
m._W = W
m._caps = caps
m._nodes_hist, m._ub_hist, m._lb_hist = (
    [],
    [],
    [],
)  # Las variables para graficar la respuesta

m.update()
m = run_bnb_with_fp_rounds(
    model=m,
    xvars=m._x,
    rounds=5,
    nodes_per_round=5000,
    collect_every_k_nodes=5,
    pool_cap=3000,
    seeds_per_round=3,
    fp_max_iters=40,
    fp_time_limit=10.0,
    inject_as_start=True,
    fp_verbose=True,
    driver_verbose=True,
)

Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Set parameter Presolve to value 0
Set parameter Symmetry to value 0
Set parameter ConcurrentMIP to value 1
Set parameter MIPFocus to value 0
Set parameter VarBranch to value 0
Set parameter NodeMethod to value 1
Set parameter BranchDir to value 0
Set parameter MIPGap to value 0
Set parameter FeasibilityTol to value 1e-06
Set parameter IntFeasTol to value 1e-05
Set parameter NumericFocus to value 0
Set parameter TimeLimit to value 60
Set parameter Threads to value 0
Set parameter Seed to value 42
Set parameter LogToConsole to value 1

=== Presolve FP: usando relajación LP raíz ===
[Presolve] FP seed 1/3  ||frac||=0.600
Corriendo feasibility pump con combinación convexa, con beta  0.7
[FP-seeded] it=0  ||x-z||_1(int)=0.6
[FP-seeded] it=1  ||x-z||_1(int)=0.6
[FP-seeded] perturbación (flips=2) en it=1.
[FP-seeded] it=2  ||x-z||_1(int)=0
[FP-seeded] solución factible en it=2.
[Presolve] FP encontró solución factible -> inyect